# Graph Construction: What's Actually Happening

This notebook answers precisely: **why are some real covalent bonds absent from the SOTA graph?**

The short answer: the SOTA graph does **not** use `molecule_graph.graph.edges()` for bond node creation. It uses the `bonds` column in the PKL, which is the **QTAIM bond critical point list** — a fundamentally different thing.

Steps:
1. Real molecular graph from `molecule_graph.graph.edges()`
2. What the `bonds` column actually contains vs real bonds
3. How `HeteroCompleteGraphFromMolWrapper` builds the HeteroData
4. Feature vectors per experiment (topo / topo_elf / topo_qtaim)
5. SOTA graph vs real bonds comparison
6. **ELF CP graph** from `build_hetero_critical_from_json` — topology, nodes, edges, tensors
7. All three side by side

## Setup

In [1]:
import json, ast, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image
import io, torch

ATOM_COLOR = {"C":"#2C2C2A","N":"#1E5FA5","O":"#C94F2A","H":"#9E9C96","F":"#1D9E75"}
BOND_COLOR  = "#C97B1A"
print("Imports OK")


Imports OK


In [2]:
BASE      = "/lustre/fsn1/projects/rech/ihj/urb54jd"
PKL_TRAIN = f"{BASE}/qtaim_embed_private/data_suba/filtered_qtaim_fullqm9/train_43k.pkl"
PKL_VAL   = f"{BASE}/qtaim_embed_private/data_suba/filtered_qtaim_fullqm9/val_43k.pkl"
PKL_TEST  = f"{BASE}/qtaim_embed_private/data_suba/filtered_qtaim_fullqm9/test_43k.pkl"
ELF_CSV   = f"{BASE}/gnn/control_and_critical_points_GNNs/data/qm9_43k_clean_with_val.csv"
JSON_DIR  = f"{BASE}/gnn/control_and_critical_points_GNNs/data/criticalpoints_jsonfiles"
print("Paths set")


Paths set


In [ ]:
print("Loading PKL...")
sota_df = pd.concat([pd.read_pickle(PKL_TRAIN),
                     pd.read_pickle(PKL_VAL),
                     pd.read_pickle(PKL_TEST)], ignore_index=True)
sota_df["gdb_num"] = sota_df["names"].str.extract(r"gdb_(\d+)\.xyz").astype(int)
elf_df  = pd.read_csv(ELF_CSV)
print(f"SOTA PKL: {len(sota_df)} rows")

# ── Bond-related columns ─────────────────────────────────────────────────────
print("\nBond-related columns in PKL:")
for c in [c for c in sota_df.columns if "bond" in c.lower()]:
    print(f"  {c}")


Loading PKL...


In [ ]:
def _parse_bonds(raw):
    s = str(raw).strip()
    if s.startswith("[["): s = s[1:-1]
    try:    return [(int(a), int(b)) for a,b in ast.literal_eval(s)]
    except: return []

def _parse_array(raw):
    s = str(raw).strip()
    if s.startswith("[[") and s.endswith("]]"): s = s[1:-1]
    try:    return np.array(ast.literal_eval(s), dtype=float)
    except: return np.array([])

BOND_QTAIM_COLS = [
    "extra_feat_bond_Hamiltonian_K","extra_feat_bond_e_density",
    "extra_feat_bond_lap_e_density","extra_feat_bond_e_loc_func",
    "extra_feat_bond_ave_loc_ion_E","extra_feat_bond_delta_g_promolecular",
    "extra_feat_bond_delta_g_hirsh","extra_feat_bond_esp_nuc",
    "extra_feat_bond_esp_e","extra_feat_bond_esp_total",
    "extra_feat_bond_grad_norm","extra_feat_bond_lap_norm",
    "extra_feat_bond_eig_hess","extra_feat_bond_det_hessian",
    "extra_feat_bond_ellip_e_dens","extra_feat_bond_eta",
    "extra_feat_bond_energy_density","extra_feat_bond_lol",
    "extra_feat_bond_Lagrangian_K",
]

ATOM_COLOR = {"C":"#2C2C2A","N":"#1E5FA5","O":"#C94F2A","H":"#9E9C96","F":"#1D9E75"}
CP_BOND_C  = "#C97B1A"   # bonding attractor — orange diamond
CP_LONE_C  = "#7B1FA2"   # lone-pair attractor — purple diamond
CP_CORE_C  = "#888888"   # core attractor — grey triangle

IMG_SIZE = 600   # pixels for the molecule image

def get_atom_pixel_coords(smiles, n_atoms, img_size=IMG_SIZE):
    """
    Get pixel coordinates of each atom as drawn by RDKit MolToImage.
    Uses rdMolDraw2D to get the EXACT same positions as the PNG rendering.
    Returns: pos dict {atom_idx: (x_pixel, y_pixel)}, PIL image
    """
    from rdkit.Chem.Draw import rdMolDraw2D
    from PIL import Image
    import io

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    for a in mol.GetAtoms():
        a.SetAtomMapNum(a.GetIdx())

    drawer = rdMolDraw2D.MolDraw2DCairo(img_size, img_size)
    drawer.drawOptions().addAtomIndices = False
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()

    # Get atom positions in normalised [0,1] coords then scale to pixels
    pos = {}
    for i in range(n_atoms):
        pt = drawer.GetDrawCoords(i)
        pos[i] = np.array([pt.x, pt.y])

    img = Image.open(io.BytesIO(drawer.GetDrawingText()))
    return pos, img


def draw_graph_on_image(ax, smiles, n_atoms, syms,
                        edges, edge_colors=None, edge_styles=None, edge_widths=None,
                        extra_nodes=None, title="", img_size=IMG_SIZE):
    """
    Draw molecular graph OVERLAID ON the RDKit structure image.
    Atom positions come from rdMolDraw2D — same as the actual rendering.
    So the graph nodes sit exactly on top of atom symbols.

    extra_nodes: list of (x, y, color, marker, markersize) in pixel coords
    """
    pos, img = get_atom_pixel_coords(smiles, n_atoms, img_size)

    # Show the molecule image as background
    ax.imshow(img, extent=[0, img_size, img_size, 0])  # y-axis: 0=top
    ax.set_xlim(0, img_size)
    ax.set_ylim(img_size, 0)   # flip so y increases downward (image convention)
    ax.set_aspect("equal")
    ax.axis("off")

    # Draw edges
    for k, (i, j) in enumerate(edges):
        ec = edge_colors[k] if edge_colors else "#FF0000"
        es = edge_styles[k]  if edge_styles  else "-"
        ew = edge_widths[k]  if edge_widths  else 2.0
        ax.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]],
                color=ec, lw=ew, ls=es, zorder=2,
                solid_capstyle="round", alpha=0.85)

    # Draw extra nodes (bond nodes, attractor nodes) at given pixel positions
    if extra_nodes:
        for (px, py), color, marker, ms in extra_nodes:
            ax.plot(px, py, marker, color=color, ms=ms, zorder=4,
                    markeredgecolor="white", markeredgewidth=0.8)

    # Draw atom node circles overlaid on atom symbols
    for i in range(n_atoms):
        c  = ATOM_COLOR.get(syms[i], "#888888")
        sz = 180 if syms[i] != "H" else 60
        ax.scatter(pos[i][0], pos[i][1], c=c, s=sz, zorder=3,
                   edgecolors="white", linewidths=0.8, alpha=0.75)
        ax.text(pos[i][0], pos[i][1], str(i),
                ha="center", va="center", fontsize=5,
                color="white", fontweight="bold", zorder=5)

    ax.set_title(title, fontsize=10, pad=6)


def mol_image_only(ax, smiles, n_atoms, title="Molecular Structure", img_size=IMG_SIZE):
    """Just show the RDKit image with atom map numbers — no overlay."""
    _, img = get_atom_pixel_coords(smiles, n_atoms, img_size)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=10)


def midpx(pos, i, j):
    """Midpoint in pixel coordinates."""
    return ((pos[i][0]+pos[j][0])/2, (pos[i][1]+pos[j][1])/2)


print("Helpers loaded — graph overlaid on RDKit image at exact atom positions")


## Pick a molecule

In [ ]:
MOL_ID = "dsgdb9nsd_021159"   # ← change me

elf_row  = elf_df[elf_df["ID"]==MOL_ID].iloc[0]
gdb_num  = int(elf_row["GDB_Index"])
smiles   = str(elf_row.get("canonical_smiles", elf_row.get("SMILES","")))
sota_row = sota_df[sota_df["gdb_num"]==gdb_num].iloc[0]

mol_pm   = sota_row["molecule"]
mol_g    = sota_row["molecule_graph"]
n_atoms  = len(mol_pm.sites)
syms     = [str(s.species.elements[0].symbol) for s in mol_pm.sites]

# ── THREE different bond lists ────────────────────────────────────────────────
# 1) molecule_graph.graph.edges() — full real covalent connectivity
mg_edges = sorted({(min(u,v),max(u,v)) for u,v in mol_g.graph.edges()})

# 2) "bonds" column in PKL — what HeteroCompleteGraphFromMolWrapper uses
#    when bond_key="bonds"  (the QTAIM bond list, NOT full connectivity)
raw_bonds = sota_row["bonds"]
if isinstance(raw_bonds[0], (list, tuple)): raw_bonds = raw_bonds[0]
bonds_col = sorted({(min(a,b),max(a,b)) for a,b in raw_bonds if a!=b})

# 3) extra_feat_bond_indices_qtaim — parallel index for feature arrays
qtaim_idx = _parse_bonds(sota_row["extra_feat_bond_indices_qtaim"])
qtaim_set = {(min(a,b),max(a,b)) for a,b in qtaim_idx if a!=b}

print(f"Molecule : {MOL_ID}  ({n_atoms} atoms)  SMILES: {smiles}")
print(f"\n1) molecule_graph.edges() : {len(mg_edges)} bonds  ← real covalent connectivity")
print(f"2) bonds column (PKL)     : {len(bonds_col)} bonds  ← used as graph topology")
print(f"3) qtaim index list       : {len(qtaim_set)} unique pairs  ← feature array index")
print(f"\nAre (1) and (2) the same? {set(mg_edges)==set(bonds_col)}")


## bonds_original — what does it contain?

The PKL has both a `bonds` column and a `bonds_original` column. We know `bonds` = QTAIM BCP list. What is `bonds_original`? It is the `map_key` used by `get_bond_features()` in `descriptors.py` to look up which position in the feature arrays each bond sits at.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# bonds_original column — what is it?
# ══════════════════════════════════════════════════════════════════════════════

raw_orig = sota_row["bonds_original"]

# Unwrap if nested list
if isinstance(raw_orig, list):
    if len(raw_orig) == 1 and isinstance(raw_orig[0], list):
        raw_orig = raw_orig[0]
    bonds_original = [(int(a), int(b)) for a,b in raw_orig if a != b]
else:
    bonds_original = _parse_bonds(raw_orig)

bonds_orig_set = {(min(a,b), max(a,b)) for a,b in bonds_original}

mg  = set(mg_edges)
bnd = set(bonds_col)      # "bonds" col = QTAIM BCPs

print(f"bonds_original column: {len(bonds_orig_set)} unique pairs")
print(f"molecule_graph.edges(): {len(mg)} pairs")
print(f"bonds column (QTAIM):   {len(bnd)} pairs")
print()
print(f"bonds_original == molecule_graph.edges() ? {bonds_orig_set == mg}")
print(f"bonds_original == bonds col (QTAIM)?      {bonds_orig_set == bnd}")
print()

only_orig = sorted(bonds_orig_set - mg)
only_mg   = sorted(mg - bonds_orig_set)

print(f"In bonds_original but NOT in molecule_graph: {len(only_orig)}")
for i,j in only_orig:
    print(f"  ({i},{j}) {syms[i]}-{syms[j]}")

print(f"In molecule_graph but NOT in bonds_original: {len(only_mg)}")
for i,j in only_mg:
    print(f"  ({i},{j}) {syms[i]}-{syms[j]}")

print()
print("bonds_original raw values:")
for b in sorted(bonds_orig_set):
    i,j = b
    in_qtaim = b in bnd
    in_mg    = b in mg
    tag = []
    if in_mg:    tag.append("✓ real bond")
    if in_qtaim: tag.append("✓ in QTAIM")
    if not in_qtaim: tag.append("✗ NOT in QTAIM bonds col")
    print(f"  ({i:2d},{j:2d}) {syms[i]}-{syms[j]}  {'  '.join(tag)}")


In [ ]:
# ── What bonds_original tells us about the SOTA graph construction ────────────
# 
# Now we know all four bond lists. Let's map out the full picture:
#   mg_edges       = molecule_graph.graph.edges() = real covalent bonds
#   bonds_original = ??? (we just found out above)
#   bonds_col      = "bonds" column = QTAIM BCP list = graph topology used
#   qtaim_set      = extra_feat_bond_indices_qtaim = feature array parallel index
#
# The critical question: which of these does HeteroCompleteGraphFromMolWrapper
# use to create bond nodes?
#
# From the config: bond_key="bonds", map_key="bonds_original" (or "bonds")
# get_bond_features() in descriptors.py:
#   bonds = row[bond_key]          ← creates bond nodes from THIS list
#   bond_index_map = row[map_key].index(bond)  ← looks up feature position
# ──────────────────────────────────────────────────────────────────────────────

mg  = set(mg_edges)
bnd = set(bonds_col)
orig = bonds_orig_set

print("="*65)
print("  COMPLETE PICTURE: all four bond lists")
print("="*65)
print()
print(f"  molecule_graph.edges()    : {len(mg):3d}  real covalent bonds")
print(f"  bonds_original column     : {len(orig):3d}  {'= real bonds' if orig==mg else '≠ real bonds — see below'}")
print(f"  bonds column (QTAIM)      : {len(bnd):3d}  QTAIM BCP list → used as bond_key")
print(f"  extra_feat_bond_indices   : {len(qtaim_set):3d}  parallel feature index")
print()
print(f"  bonds col == qtaim index? {bnd == qtaim_set}  (should be True — same list)")
print(f"  bonds_original == mg?     {orig == mg}")
print()

# Show which real bonds are absent from the QTAIM bond node list
missing_from_sota = sorted(mg - bnd)
phantom_in_sota   = sorted(bnd - mg)

print(f"  Real bonds with NO bond node in SOTA graph: {len(missing_from_sota)}")
for i,j in missing_from_sota:
    in_orig = (min(i,j),max(i,j)) in orig
    print(f"    ({i:2d},{j:2d}) {syms[i]}-{syms[j]}  [in bonds_original: {in_orig}]")

print()
print(f"  Phantom bond nodes (QTAIM-only, not real bonds): {len(phantom_in_sota)}")
for i,j in phantom_in_sota:
    in_orig = (min(i,j),max(i,j)) in orig
    print(f"    ({i:2d},{j:2d}) {syms[i]}-{syms[j]}  [in bonds_original: {in_orig}]")

print()
print("  CONCLUSION:")
print("  bonds_original is the map_key — used by get_bond_features() to")
print("  find which position in the feature array each bond sits at.")
print("  bond_key='bonds' (QTAIM list) creates the actual bond nodes.")
print("  This confirms: SOTA graph topology = QTAIM BCP topology, NOT")
print("  the full molecular bond graph.")
print("="*65)


## Step 1 — Real molecular graph

`molecule_graph.graph.edges()` gives the full covalent connectivity from pymatgen. This is what a standard GNN would use.

In [ ]:
# Step 1: Real molecular graph
# Left: RDKit image only  |  Right: graph overlaid on same image
pos, _ = get_atom_pixel_coords(smiles, n_atoms)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

mol_image_only(axes[0], smiles, n_atoms,
               title=f"Molecular Structure\n{n_atoms} atoms, {len(mg_edges)} real bonds")

draw_graph_on_image(
    axes[1], smiles, n_atoms, syms, mg_edges,
    edge_colors=["#444444"]*len(mg_edges),
    edge_widths=[2.5]*len(mg_edges),
    title=f"Graph Representation\n{n_atoms} atom nodes, {len(mg_edges)} edges"
)

plt.tight_layout(); plt.show()

print("Real bonds (molecule_graph.graph.edges()):")
for i,j in mg_edges:
    print(f"  ({i:2d},{j:2d})  {syms[i]}-{syms[j]}")


## Step 2 — The `bonds` column is NOT the molecular graph

The config uses `bond_key='bonds'`, which points to the `bonds` column in the PKL. This column contains the **QTAIM BCP list** — not the full covalent bonds.

Some real bonds are absent. Some phantom non-covalent BCPs are present.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — What is in the "bonds" column?
#
# In HeteroCompleteGraphFromMolWrapper the graph is built using:
#   bond_key = "bonds"   (config["dataset"]["bond_key"])
#
# This means the BOND NODES are created from row["bonds"], NOT from
# molecule_graph.graph.edges(). Let us see what "bonds" actually contains.
# ══════════════════════════════════════════════════════════════════════════════
mg  = set(mg_edges)
bnd = set(bonds_col)

only_mg   = sorted(mg  - bnd)   # real bonds NOT in the graph
only_bond = sorted(bnd - mg)    # graph bonds NOT real covalent bonds
in_both   = sorted(mg  & bnd)

print("Comparing molecule_graph.edges() vs PKL 'bonds' column:")
print(f"  In both (real & in graph)        : {len(in_both)}")
print(f"  Only in mol_graph (MISSING)      : {len(only_mg)}  ← not included as bond nodes!")
print(f"  Only in bonds col (PHANTOM)      : {len(only_bond)}  ← bond nodes for non-real pairs!")
print()
print(f"Real bonds that ARE bond nodes ({len(in_both)}):")
for i,j in in_both:
    print(f"  ({i:2d},{j:2d})  {syms[i]}-{syms[j]}")
print(f"\nReal bonds MISSING from graph ({len(only_mg)}) — NO bond node created:")
for i,j in only_mg:
    print(f"  ({i:2d},{j:2d})  {syms[i]}-{syms[j]}  ← this bond has NO node in the graph")
print(f"\nPHANTOM bond nodes ({len(only_bond)}) — not real covalent bonds:")
for i,j in only_bond:
    print(f"  ({i:2d},{j:2d})  {syms[i]}-{syms[j]}  ← QTAIM BCP, not a covalent bond")


In [ ]:
# Step 2: real vs SOTA vs difference — all overlaid on structure image
pos, _ = get_atom_pixel_coords(smiles, n_atoms)
mg  = set(mg_edges); bnd = set(bonds_col)
in_both   = sorted(mg & bnd)
only_mg   = sorted(mg - bnd)    # real bonds with NO bond node in SOTA
only_bond = sorted(bnd - mg)    # phantom QTAIM BCPs in SOTA

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Panel 1: full real molecular graph
draw_graph_on_image(axes[0], smiles, n_atoms, syms, sorted(mg_edges),
    edge_colors=["#444444"]*len(mg_edges),
    edge_widths=[2.5]*len(mg_edges),
    title=f"Real molecular graph\n{len(mg)} real bonds (molecule_graph.edges)")

# Panel 2: SOTA graph — bond nodes as squares at midpoints
sota_edges  = sorted(bonds_col)
sota_ec     = ["#2E7D32" if e in mg else "#C62828" for e in sota_edges]
sota_extra  = [(midpx(pos,i,j), "#2E7D32" if (i,j) in mg else "#C62828", "s", 11)
               for i,j in sota_edges]
draw_graph_on_image(axes[1], smiles, n_atoms, syms, sota_edges,
    edge_colors=sota_ec,
    edge_widths=[2.5]*len(sota_edges),
    extra_nodes=sota_extra,
    title=f"SOTA graph (QTAIM BCP topology)\n"
          f"{len(sota_edges)} bond nodes  ·  {len(in_both)} real  ·  {len(only_bond)} phantom")

# Panel 3: difference overlay
all_e  = in_both + only_mg + only_bond
all_c  = ["#2E7D32"]*len(in_both) + ["#E65100"]*len(only_mg) + ["#C62828"]*len(only_bond)
all_ls = ["-"]*len(in_both) + ["--"]*len(only_mg) + [":"]*len(only_bond)
all_lw = [2.5]*len(in_both) + [2.0]*len(only_mg) + [2.0]*len(only_bond)
draw_graph_on_image(axes[2], smiles, n_atoms, syms, all_e,
    edge_colors=all_c, edge_styles=all_ls, edge_widths=all_lw,
    title=f"Difference\ngreen=real+SOTA  ·  orange--=missing ({len(only_mg)})  ·  red:=phantom ({len(only_bond)})")

leg = [
    Line2D([0],[0],color="#2E7D32",lw=2.5,label=f"real bond in SOTA ({len(in_both)})"),
    Line2D([0],[0],color="#E65100",lw=2,ls="--",label=f"real bond MISSING from SOTA ({len(only_mg)})"),
    Line2D([0],[0],color="#C62828",lw=2,ls=":",label=f"phantom QTAIM BCP ({len(only_bond)})"),
    plt.scatter([],[],marker="s",c="#2E7D32",s=80,label="SOTA bond node (real)"),
    plt.scatter([],[],marker="s",c="#C62828",s=80,label="SOTA bond node (phantom)"),
]
fig.legend(handles=leg, loc="lower center", ncol=3, fontsize=9,
           bbox_to_anchor=(0.5,-0.02))
fig.suptitle(f"{MOL_ID}  |  {smiles}", fontsize=11)
plt.tight_layout(); plt.show()


## Step 3 — HeteroData construction

Tracing exactly how `HeteroCompleteGraphFromMolWrapper` builds the graph from the `bonds` column.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — How this graph is physically constructed
#
# HeteroCompleteGraphFromMolWrapper with bond_key="bonds":
#   for each bond in row["bonds"]:
#       create a bond NODE
#       add edges: atom_i → bond_node, atom_j → bond_node (and reverse)
#   NOT using molecule_graph.graph.edges() for bond nodes
#
# Node types: atom (N), bond (B), global (1)
# Edge types:  atom→bond (a2b), bond→atom (b2a),
#              atom→global (a2g), global→atom (g2a),
#              bond→global (b2g), global→bond (g2b),
#              self-loops on atom and bond
# ══════════════════════════════════════════════════════════════════════════════

B = len(bonds_col)   # number of bond nodes (from "bonds" col, NOT mol_graph)
N = n_atoms

a2b_src, a2b_dst = [], []
b2a_src, b2a_dst = [], []
for bond_idx, (i,j) in enumerate(bonds_col):
    a2b_src += [i, j];          a2b_dst += [bond_idx, bond_idx]
    b2a_src += [bond_idx]*2;    b2a_dst += [i, j]

a2g_src = list(range(N)); a2g_dst = [0]*N
b2g_src = list(range(B)); b2g_dst = [0]*B

print("SOTA HeteroData graph tensors:")
print(f"  atom nodes  : {N}")
print(f"  bond nodes  : {B}  (from bonds col — QTAIM BCPs)")
print(f"  global nodes: 1")
print()
print(f"  (atom,a2b,bond)   edge_index shape: [2, {len(a2b_src)}]")
print(f"    src (atom idx): {a2b_src}")
print(f"    dst (bond idx): {a2b_dst}")
print()
print(f"  (bond,b2a,atom)   edge_index shape: [2, {len(b2a_src)}]")
print(f"    src (bond idx): {b2a_src}")
print(f"    dst (atom idx): {b2a_dst}")
print()
print(f"  (atom,a2g,global) src={a2g_src}, dst={a2g_dst}")
print(f"  (bond,b2g,global) src={b2g_src}, dst={b2g_dst}")
print()
print("Bond node table (what each bond node represents):")
for k,(i,j) in enumerate(bonds_col):
    is_real = (i,j) in set(mg_edges)
    tag = "✓ real bond" if is_real else "✗ PHANTOM (not a covalent bond)"
    print(f"  bond_node {k:2d}  atoms ({i:2d},{j:2d})  {syms[i]}-{syms[j]}  {tag}")


## Step 4 — Feature vectors per experiment

All three experiments (topo, topo_elf, topo_qtaim) use the same QTAIM topology. Only the feature values on bond nodes differ.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Feature vectors on bond nodes (what changes between experiments)
#
# Exp A topo:       bond.feat = [metal_bond, ring_incl, ring_3..7]    = 7
# Exp B topo_elf:   bond.feat = [7 topo + 5 ELF]                     = 12
# Exp C topo_qtaim: bond.feat = [7 topo + 19 QTAIM]                  = 26
#
# The 19 QTAIM features come from extra_feat_bond_* columns.
# The lookup works as: position k in those arrays corresponds to bond qtaim_idx[k]
# ══════════════════════════════════════════════════════════════════════════════

# Build QTAIM feature lookup: (i,j) → np.array(19)
qtaim_pairs = _parse_bonds(sota_row["extra_feat_bond_indices_qtaim"])
feat_arrays  = {c: _parse_array(sota_row[c]) for c in BOND_QTAIM_COLS}
feat_lookup  = {}
for k,(i,j) in enumerate(qtaim_pairs):
    key = (min(i,j),max(i,j))
    feat_lookup[key] = np.array([
        feat_arrays[c][k] if k<len(feat_arrays[c]) else 0. for c in BOND_QTAIM_COLS])

print("Exp C (topo+QTAIM) — bond.feat [26] for each bond node:")
print(f"  Columns 0-6  : topology  [metal_bond, ring_incl, ring_3, ring_4, ring_5, ring_6, ring_7]")
print(f"  Columns 7-25 : QTAIM     [{', '.join([c.replace('extra_feat_bond_','') for c in BOND_QTAIM_COLS[:3]])}...]")
print()
for k,(i,j) in enumerate(bonds_col):
    is_real = (i,j) in set(mg_edges)
    has_q   = (i,j) in feat_lookup
    tag     = ("✓ real" if is_real else "✗ phantom") + ("  has QTAIM" if has_q else "  zeros(19)")
    f = feat_lookup.get((i,j), np.zeros(19))
    print(f"  bond_node {k:2d} ({i},{j}) {syms[i]}-{syms[j]}  {tag}")
    if has_q:
        print(f"    QTAIM feat[0:3] = {[round(v,5) for v in f[:3]]}")
    else:
        print(f"    QTAIM feat[0:3] = [0.0, 0.0, 0.0]  ← no BCP found for this pair")


## Step 5 — Full comparison

In [ ]:
# Step 5: Full comparison — structure + SOTA + ELF
pos, _ = get_atom_pixel_coords(smiles, n_atoms)
mg  = set(mg_edges); bnd = set(bonds_col)
in_both = sorted(mg & bnd); only_mg = sorted(mg-bnd); only_bond = sorted(bnd-mg)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

mol_image_only(axes[0], smiles, n_atoms,
    title=f"Molecular Structure\n{n_atoms} atoms, {len(mg)} real bonds")

# SOTA graph
sota_e  = sorted(bonds_col)
sota_c  = ["#2E7D32" if e in mg else "#C62828" for e in sota_e]
sota_ex = [(midpx(pos,i,j), "#2E7D32" if (i,j) in mg else "#C62828", "s", 11)
           for i,j in sota_e]
draw_graph_on_image(axes[1], smiles, n_atoms, syms, sota_e,
    edge_colors=sota_c, edge_widths=[2.5]*len(sota_e), extra_nodes=sota_ex,
    title=f"SOTA graph (QTAIM BCP topology)\n"
          f"{n_atoms} atom + {len(sota_e)} bond nodes (squares)\n"
          f"{len(in_both)} real  ·  {len(only_bond)} phantom  ·  {len(only_mg)} missing")

# ELF graph — bonding attractors only
elf_e  = []
elf_c  = []
elf_ex = []
for m in cp_metadata:
    if m["type"]=="valence" and len(m["atoms"])==2:
        i,j = m["atoms"][0], m["atoms"][1]
        real = (min(i,j),max(i,j)) in mg
        elf_e.append((i,j))
        elf_c.append("#2E7D32" if real else "#C62828")
        elf_ex.append((midpx(pos,i,j), CP_BOND_C, "D", 10))

draw_graph_on_image(axes[2], smiles, n_atoms, syms, elf_e,
    edge_colors=elf_c, edge_widths=[2.5]*len(elf_e), extra_nodes=elf_ex,
    title=f"ELF graph (bonding attractors)\n"
          f"{n_atoms} atom + {len(elf_e)} attractor nodes (diamonds)\n"
          f"topology from JSON Atom list")

leg = [
    mpatches.Patch(color=ATOM_COLOR["C"],label="C"),
    mpatches.Patch(color=ATOM_COLOR["N"],label="N"),
    mpatches.Patch(color=ATOM_COLOR["O"],label="O"),
    mpatches.Patch(color=ATOM_COLOR["H"],label="H"),
    Line2D([0],[0],color="#2E7D32",lw=2.5,label="real covalent bond"),
    Line2D([0],[0],color="#C62828",lw=2,label="phantom / non-bonded"),
    Line2D([0],[0],color="#E65100",lw=2,ls="--",label=f"real bond missing from SOTA ({len(only_mg)})"),
    plt.scatter([],[],marker="s",c="#2E7D32",s=80,label="SOTA bond node (real)"),
    plt.scatter([],[],marker="s",c="#C62828",s=80,label="SOTA bond node (phantom)"),
    plt.scatter([],[],marker="D",c=CP_BOND_C,s=80,label="ELF bonding attractor"),
]
fig.legend(handles=leg, loc="lower center", ncol=5, fontsize=8,
           bbox_to_anchor=(0.5,-0.04))
fig.suptitle(
    f"{MOL_ID}  |  {smiles}\n"
    f"Real: {len(mg)} bonds  ·  "
    f"SOTA: {len(sota_e)} nodes ({len(only_bond)} phantom, {len(only_mg)} missing)  ·  "
    f"ELF: {len(elf_e)} attractor nodes",
    fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY: What this means
# ══════════════════════════════════════════════════════════════════════════════

mg  = set(mg_edges); bnd = set(bonds_col)

print("="*65)
print("  SUMMARY — How SOTA graph differs from real molecular graph")
print("="*65)
print()
print(f"  Real molecular bonds (molecule_graph.edges) : {len(mg)}")
print(f"  Bond nodes in SOTA graph  (bonds column)    : {len(bnd)}")
print()
print(f"  Real bonds that ARE in SOTA graph            : {len(mg & bnd)}")
print(f"  Real bonds MISSING from SOTA graph           : {len(mg-bnd)}")
print(f"    → no bond node created for these bonds")
print(f"    → atom pair has NO direct path through a bond node")
print(f"  PHANTOM bond nodes (not real covalent bonds) : {len(bnd-mg)}")
print(f"    → QTAIM non-bonded BCPs treated as bonds")
print()
print("  WHY: bond_key='bonds' in the config points to the QTAIM BCP list,")
print("  not to molecule_graph.graph.edges(). This is intentional in the")
print("  SOTA paper — the graph topology IS the QTAIM topology of ρ(r).")
print()
print("  CONSEQUENCE for Exp A/B/C:")
print("  All three experiments use this SAME QTAIM topology as the graph.")
print("  The 'bonds' column is always used for bond_key.")
print("  molecule_graph.graph.edges() is NOT used for graph construction.")
print()
print("  THIS IS WHY bonds appear 'missing':")
print("  They are missing from the QTAIM BCP list, so they have")
print("  no bond node at all — not just zeros, but literally absent.")
print()
print("  Your ELF CP graph avoids this: every real bond has a CP node")
print("  (because ELF basins are attributed to all real bond pairs),")
print("  and the graph topology comes from electron pair domains, not")
print("  from QTAIM saddle points of ρ(r).")
print("="*65)


## Step 6 — Your ELF CP graph (`build_hetero_critical_from_json`)

This graph is built from the **JSON critical points file**, not from the PKL at all.

Key facts from `hetero_utils.py`:
- **Atom nodes**: one per atom in `data["Atoms"]` → `atom.x = [z, 0, 0, 0, 0]` (5 features)
- **CP nodes**: one per critical point in `data["Critical Points"]` → `cp.x = [group_value, volume, population, charge, value]` (5 features)
- **ALL CPs** are included: bonding valence (between 2 atoms), lone-pair valence (1 atom), and core (1 atom)
- **Edges** come from the `Atom list` field of each CP → connect that CP node to each atom in its list
- Edge direction: `cp → atom` (and reverse `atom → cp` if `edge_mode="bidir"`)

**Crucially**: the graph topology (which atom pairs are connected via a CP) comes from ELF basin attribution — which basins are shared between which atoms — not from QTAIM BCPs.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — ELF CP graph construction traced from build_hetero_critical_from_json
# ══════════════════════════════════════════════════════════════════════════════

jf = f"{JSON_DIR}/integrated_aimel_{gdb_num:06d}.json"
with open(jf) as f: jdata = json.load(f)

atoms_json = jdata["Atoms"]
cps_json   = jdata["Critical Points"]

# ── Atom nodes (same as build_hetero_critical_from_json) ─────────────────────
atom_list_map = {}
atom_node_feats = []
for idx, (_, info) in enumerate(atoms_json.items()):
    z = info.get("z value", 0)
    atom_node_feats.append([z, 0, 0, 0, 0])
    atom_list_map[str(info.get("Atom list","")).strip()] = idx

# ── CP nodes (ALL CPs — bonding, lone pair, core) ────────────────────────────
cp_keys = sorted(cps_json.keys())
cp_node_feats = []
cp_metadata   = []   # for display
for k in cp_keys:
    cp = cps_json[k]
    group       = cp.get("group", 0)
    group_value = 0 if group==0 else -3
    volume      = cp.get("volume", 0)
    population  = cp.get("population", 0)
    charge      = cp.get("charge", 0)
    value       = cp.get("value", 0)
    cp_node_feats.append([group_value, volume, population, charge, value])

    # parse atom list for edge construction
    raw   = str(cp.get("Atom list","")).strip()
    items = [a.strip() for a in raw.strip("()").split(",") if a.strip()]
    idxs  = [atom_list_map[a] for a in items if a in atom_list_map]
    cp_type = cp.get("Type","?")
    cp_metadata.append({"key": k, "type": cp_type, "atoms": idxs,
                         "volume": volume, "population": population,
                         "charge": charge, "eta": value, "group": group})

# ── Edges: CP → atom (from Atom list) ────────────────────────────────────────
cp_to_atom_pairs = []
for cp_idx, k in enumerate(cp_keys):
    cp = cps_json[k]
    raw   = str(cp.get("Atom list","")).strip()
    items = [a.strip() for a in raw.strip("()").split(",") if a.strip()]
    for a in items:
        ai = atom_list_map.get(a)
        if ai is not None:
            cp_to_atom_pairs.append((cp_idx, ai))

import torch
n_atoms_json = len(atoms_json)
n_cps        = len(cp_keys)

# Classify CPs
valence_bond = [m for m in cp_metadata if m["type"]=="valence" and len(m["atoms"])==2]
valence_lone = [m for m in cp_metadata if m["type"]=="valence" and len(m["atoms"])==1]
core_cps     = [m for m in cp_metadata if m["type"]=="core"]

print(f"ELF JSON: {n_atoms_json} atoms, {n_cps} total CP nodes")
print(f"  Valence bonding CPs (2 atoms)  : {len(valence_bond)}  ← connect two atoms")
print(f"  Valence lone-pair CPs (1 atom) : {len(valence_lone)}  ← connect to one atom")
print(f"  Core CPs (1 atom)              : {len(core_cps)}   ← connect to one atom")
print()
print(f"Edges: {len(cp_to_atom_pairs)} (cp→atom connections)")
print()

# Compare bonding CP atom pairs vs real bonds
mg = set(mg_edges)
elf_bond_pairs = {(min(m["atoms"][0],m["atoms"][1]),
                   max(m["atoms"][0],m["atoms"][1]))
                  for m in valence_bond}
elf_only = sorted(elf_bond_pairs - mg)
mg_only  = sorted(mg - elf_bond_pairs)
both     = sorted(elf_bond_pairs & mg)

print(f"Bonding CP pairs vs real molecular bonds:")
print(f"  In both (real & has bonding CP)          : {len(both)}")
print(f"  Real bonds with NO bonding CP (ELF miss) : {len(mg_only)}")
print(f"  Bonding CPs not in mol_graph             : {len(elf_only)}")
print()
if mg_only:
    print(f"  Real bonds without a bonding CP:")
    for i,j in mg_only:
        print(f"    ({i:2d},{j:2d})  {syms[i]}-{syms[j]}")
if elf_only:
    print(f"  ELF bonding CPs not real bonds (non-covalent basins):")
    for i,j in elf_only:
        print(f"    ({i:2d},{j:2d})  {syms[i]}-{syms[j]}")


In [ ]:
# ELF attractor graph overlaid on molecular structure
pos, _ = get_atom_pixel_coords(smiles, n_atoms)
mg = set(mg_edges)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: bonding attractors only (the edges in your model)
bond_attr = [m for m in cp_metadata if m["type"]=="valence" and len(m["atoms"])==2]
lone_attr = [m for m in cp_metadata if m["type"]=="valence" and len(m["atoms"])==1]
core_attr = [m for m in cp_metadata if m["type"]=="core"]

elf_e1 = [(m["atoms"][0], m["atoms"][1]) for m in bond_attr]
elf_c1 = ["#2E7D32" if (min(m["atoms"][0],m["atoms"][1]),
                        max(m["atoms"][0],m["atoms"][1])) in mg
           else "#C62828" for m in bond_attr]
elf_x1 = [(midpx(pos, m["atoms"][0], m["atoms"][1]), CP_BOND_C, "D", 10)
           for m in bond_attr]

draw_graph_on_image(axes[0], smiles, n_atoms, syms, elf_e1,
    edge_colors=elf_c1, edge_widths=[2.5]*len(elf_e1), extra_nodes=elf_x1,
    title=f"ELF graph — bonding attractors only\n"
          f"{n_atoms} atom nodes + {len(bond_attr)} bonding attractor nodes (◆)")

# Panel 2: all attractors (bonding + lone + core) — as actually built by the code
elf_e2  = [(m["atoms"][0], m["atoms"][1]) for m in bond_attr]  # only bonding has edges
elf_c2  = elf_c1[:]
elf_lw2 = [2.5]*len(elf_e2)
all_extra = elf_x1[:]  # bonding attractors at midpoints

# Lone-pair attractors — offset from their atom
for k, m in enumerate(lone_attr):
    i = m["atoms"][0]
    angle = (k * 137.5) % 360
    r = 0.12 * IMG_SIZE / 6   # ~12 pixels offset
    px = pos[i][0] + r * np.cos(np.radians(angle))
    py = pos[i][1] + r * np.sin(np.radians(angle))
    all_extra.append(((px, py), CP_LONE_C, "D", 8))

# Core attractors — offset from their atom (different angle)
for k, m in enumerate(core_attr):
    i = m["atoms"][0]
    angle = (k * 97.3 + 45) % 360
    r = 0.08 * IMG_SIZE / 6
    px = pos[i][0] + r * np.cos(np.radians(angle))
    py = pos[i][1] + r * np.sin(np.radians(angle))
    all_extra.append(((px, py), CP_CORE_C, "^", 7))

draw_graph_on_image(axes[1], smiles, n_atoms, syms, elf_e2,
    edge_colors=elf_c2, edge_widths=elf_lw2, extra_nodes=all_extra,
    title=f"ELF graph — all attractors (as built)\n"
          f"{n_atoms} atom + {len(bond_attr)} bonding (◆) + "
          f"{len(lone_attr)} lone-pair (◆) + {len(core_attr)} core (▲)")

leg = [
    mpatches.Patch(color=ATOM_COLOR["C"],label="C"),
    mpatches.Patch(color=ATOM_COLOR["N"],label="N"),
    mpatches.Patch(color=ATOM_COLOR["O"],label="O"),
    mpatches.Patch(color=ATOM_COLOR["H"],label="H"),
    Line2D([0],[0],marker="D",color="w",markerfacecolor=CP_BOND_C,ms=9,label="bonding attractor (◆)"),
    Line2D([0],[0],marker="D",color="w",markerfacecolor=CP_LONE_C,ms=8,label="lone-pair attractor (◆)"),
    Line2D([0],[0],marker="^",color="w",markerfacecolor=CP_CORE_C,ms=8,label="core attractor (▲)"),
    Line2D([0],[0],color="#2E7D32",lw=2.5,label="bonding edge (real bond)"),
    Line2D([0],[0],color="#C62828",lw=2.5,label="bonding edge (non-covalent)"),
]
fig.legend(handles=leg, loc="lower center", ncol=5, fontsize=8,
           bbox_to_anchor=(0.5,-0.04))
fig.suptitle(f"ELF attractor graph — {MOL_ID}  |  {smiles}", fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# ── ELF graph as PyG tensors ──────────────────────────────────────────────────
import torch

atom_x = torch.tensor(atom_node_feats, dtype=torch.float)  # [N, 5]
cp_x   = torch.tensor(cp_node_feats,   dtype=torch.float)  # [M, 5]

# Edge index (cp → atom), bidir adds reverse
cp_src = [p[0] for p in cp_to_atom_pairs]
at_dst = [p[1] for p in cp_to_atom_pairs]

ei_cp_to_atom = torch.tensor([cp_src, at_dst], dtype=torch.long)
ei_atom_to_cp = torch.tensor([at_dst, cp_src], dtype=torch.long)  # bidir

print("ELF CP HeteroData tensors (build_hetero_critical_from_json):")
print(f"  atom.x  : {list(atom_x.shape)}  [z, 0, 0, 0, 0] per atom")
print(f"  cp.x    : {list(cp_x.shape)}  [group_val, vol, pop, charge, eta] per CP")
print()
print(f"  (cp, to_atom, atom)  edge_index : [2, {len(cp_src)}]")
print(f"    src (cp idx) : {cp_src}")
print(f"    dst (atom)   : {at_dst}")
print()
print(f"  (atom, to_cp, cp)    edge_index : [2, {len(at_dst)}]  (bidir reverse)")
print()
print("CP node table (all CPs, as built by the code):")
print(f"  {'cp_idx':7}  {'type':8}  {'atoms':12}  {'group_val':10}  {'vol':7}  {'pop':7}  {'charge':8}  {'eta':7}")
print(f"  {'─'*75}")
for idx, m in enumerate(cp_metadata):
    gv = -3 if m["group"]!=0 else 0
    atoms_str = str(m["atoms"])
    print(f"  cp_{idx:3d}   {m['type']:<8}  {atoms_str:<12}  {gv:10d}  "
          f"{m['volume']:7.3f}  {m['population']:7.3f}  {m['charge']:8.3f}  {m['eta']:7.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ELF CP graph — tensor inspection + visualisation
# Reproduces the style from your original ELF notebook:
#   1. Node feature tensor (atoms then CPs, as built by the dataloader)
#   2. Edge index tensor
#   3. Edge meaning table (which CP connects to which atom and why)
#   4. Graph drawn at RDKit 2D coordinates (NOT spring layout)
# ══════════════════════════════════════════════════════════════════════════════
import torch
import numpy as np

# ── Rebuild atom.x and cp.x exactly as build_hetero_critical_from_json does ──
atom_x_raw = [[info.get("z value", 0), 0, 0, 0, 0]
               for _, info in jdata["Atoms"].items()]

cp_keys_sorted = sorted(jdata["Critical Points"].keys())
cp_x_raw = []
for k in cp_keys_sorted:
    cp = jdata["Critical Points"][k]
    group       = cp.get("group", 0)
    group_value = 0 if group == 0 else -3
    cp_x_raw.append([group_value,
                      cp.get("volume", 0),
                      cp.get("population", 0),
                      cp.get("charge", 0),
                      cp.get("value", 0)])

atom_x = torch.tensor(atom_x_raw, dtype=torch.float)
cp_x   = torch.tensor(cp_x_raw,   dtype=torch.float)

# ── Edge index (cp → atom) ────────────────────────────────────────────────────
# Rebuild atom_list_map from JSON
atom_list_map = {str(info.get("Atom list","")).strip(): idx
                 for idx, (_, info) in enumerate(jdata["Atoms"].items())}

ei_src, ei_dst = [], []   # cp_idx → atom_idx
for cp_idx, k in enumerate(cp_keys_sorted):
    cp_info = jdata["Critical Points"][k]
    raw     = str(cp_info.get("Atom list","")).strip()
    items   = [a.strip() for a in raw.strip("()").split(",") if a.strip()]
    for a in items:
        ai = atom_list_map.get(a)
        if ai is not None:
            ei_src.append(cp_idx)
            ei_dst.append(ai)

edge_index = torch.tensor([ei_src, ei_dst], dtype=torch.long)

# ── Print tensors ─────────────────────────────────────────────────────────────
print("Node features — atom.x  [z, 0, 0, 0, 0] per atom:")
print(f"  Shape: {list(atom_x.shape)}")
for i, row in enumerate(atom_x_raw):
    print(f"  atom {i:2d} ({syms[i]}): {row}")

print()
print("Node features — cp.x  [group_val, vol, pop, charge, eta] per CP:")
print(f"  Shape: {list(cp_x.shape)}")
for i, row in enumerate(cp_x_raw):
    t = cp_metadata[i]["type"]
    a = cp_metadata[i]["atoms"]
    label = f"{t} {a}"
    print(f"  cp_{i:3d} {label:<22}: [{row[0]:5.1f}, {row[1]:8.3f}, {row[2]:7.3f}, {row[3]:8.3f}, {row[4]:7.3f}]")

print()
print(f"edge_index (cp → atom):  Shape: {list(edge_index.shape)}")
print(f"  src (cp):   {ei_src}")
print(f"  dst (atom): {ei_dst}")

# ── Edge meaning table ────────────────────────────────────────────────────────
print()
print(f"  {'edge':6}  {'[cp→atom]':12}  {'CP type':8}  {'atoms':12}  {'meaning'}")
print(f"  {'─'*72}")
for k, (s, d) in enumerate(zip(ei_src, ei_dst)):
    t   = cp_metadata[s]["type"]
    a   = cp_metadata[s]["atoms"]
    if len(a) == 2:
        meaning = f"bonding basin between atom {a[0]} ({syms[a[0]]}) and atom {a[1]} ({syms[a[1]]})"
    elif t == "valence":
        meaning = f"lone-pair basin on atom {a[0]} ({syms[a[0]]})"
    else:
        meaning = f"core basin on atom {a[0]} ({syms[a[0]]})"
    print(f"  [{k:3d}]    [{s:2d} → {d:2d}]     {t:8}  {str(a):12}  → atom {d} ({syms[d]}): {meaning}")

# ── Graph drawn at RDKit 2D coordinates ──────────────────────────────────────
# Node numbering: atoms 0..N-1, then CPs 0..M-1 (shown as cp_idx offset by N)
N = len(atom_x_raw)
M = len(cp_x_raw)
pos_rdkit, _ = get_rdkit_pos(smiles, N)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: molecular structure for reference
draw_mol_structure(smiles, axes[0],
                   title=f"Molecular Structure (reference)\n{smiles}")

# Right: ELF CP graph at RDKit 2D coords
ax = axes[1]

# Compute CP positions at bond midpoints (bonding CPs) or near atom (lone/core)
cp_pos = {}
for cp_idx, m in enumerate(cp_metadata):
    a = m["atoms"]
    if len(a) == 2:
        i, j = a[0], a[1]
        cp_pos[cp_idx] = np.array([(pos_rdkit[i][0]+pos_rdkit[j][0])/2,
                                    (pos_rdkit[i][1]+pos_rdkit[j][1])/2])
    else:
        ai = a[0]
        # small random-ish offset so lone/core CPs don't overlap atom
        angle = (cp_idx * 137.5) % 360  # golden angle spread
        r = 0.35
        cp_pos[cp_idx] = np.array([pos_rdkit[ai][0] + r*np.cos(np.radians(angle)),
                                    pos_rdkit[ai][1] + r*np.sin(np.radians(angle))])

# Set axis limits based on all node positions
all_x = [pos_rdkit[i][0] for i in range(N)] + [cp_pos[k][0] for k in range(M)]
all_y = [pos_rdkit[i][1] for i in range(N)] + [cp_pos[k][1] for k in range(M)]
xpad = (max(all_x)-min(all_x))*0.2 + 0.5
ypad = (max(all_y)-min(all_y))*0.2 + 0.5
ax.set_xlim(min(all_x)-xpad, max(all_x)+xpad)
ax.set_ylim(min(all_y)-ypad, max(all_y)+ypad)
ax.set_aspect("equal"); ax.axis("off")

# Draw edges (cp → atom) — coloured by CP type
for s, d in zip(ei_src, ei_dst):
    t = cp_metadata[s]["type"]
    a = cp_metadata[s]["atoms"]
    if t == "core":
        ec, lw, ls = "#37474F", 0.8, ":"
    elif len(a) == 1:
        ec, lw, ls = CP_LONE_C, 1.0, "--"
    else:
        real = (min(a[0],a[1]), max(a[0],a[1])) in set(mg_edges)
        ec   = "#2E7D32" if real else "#C62828"
        lw, ls = 1.8, "-"
    ax.plot([cp_pos[s][0], pos_rdkit[d][0]],
            [cp_pos[s][1], pos_rdkit[d][1]],
            color=ec, lw=lw, ls=ls, zorder=1)

# Draw CP nodes
for cp_idx, m in enumerate(cp_metadata):
    t = m["type"]; a = m["atoms"]
    if t == "core":
        c, mk, ms = CP_CORE_C, "^", 7
    elif len(a) == 1:
        c, mk, ms = CP_LONE_C, "D", 8
    else:
        c, mk, ms = CP_BOND_C, "D", 9
    ax.plot(*cp_pos[cp_idx], mk, color=c, ms=ms, zorder=4,
            markeredgecolor="white", markeredgewidth=0.5)
    ax.text(cp_pos[cp_idx][0], cp_pos[cp_idx][1]+0.12,
            f"cp{cp_idx}", fontsize=4, ha="center", color="#555", zorder=5)

# Draw atom nodes
for i in range(N):
    c  = ATOM_COLOR.get(syms[i], "#888")
    sz = 350 if syms[i] != "H" else 130
    ax.scatter(*pos_rdkit[i], c=c, s=sz, zorder=3,
               edgecolors="white", linewidths=0.7)
    ax.text(pos_rdkit[i][0], pos_rdkit[i][1],
            f"{i}\n{syms[i]}", ha="center", va="center",
            fontsize=5, color="white", fontweight="bold", zorder=5)

ax.set_title(
    f"ELF CP graph — {n_atoms} atom nodes + {M} CP nodes\n"
    f"{len([m for m in cp_metadata if len(m['atoms'])==2 and m['type']=='valence'])} bonding CPs  "
    f"{len([m for m in cp_metadata if len(m['atoms'])==1 and m['type']=='valence'])} lone-pair CPs  "
    f"{len([m for m in cp_metadata if m['type']=='core'])} core CPs\n"
    f"edge_index shape: {list(edge_index.shape)}",
    fontsize=10)

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
leg = [
    mpatches.Patch(color=ATOM_COLOR["C"], label="C atom"),
    mpatches.Patch(color=ATOM_COLOR["N"], label="N atom"),
    mpatches.Patch(color=ATOM_COLOR["O"], label="O atom"),
    mpatches.Patch(color=ATOM_COLOR["H"], label="H atom"),
    Line2D([0],[0],marker="D",color="w",markerfacecolor=CP_BOND_C,ms=9,label="bonding CP (◆)"),
    Line2D([0],[0],marker="D",color="w",markerfacecolor=CP_LONE_C,ms=8,label="lone-pair CP (◆)"),
    Line2D([0],[0],marker="^",color="w",markerfacecolor=CP_CORE_C,ms=8,label="core CP (▲)"),
    Line2D([0],[0],color="#2E7D32",lw=2,label="bonding CP edge (real bond)"),
    Line2D([0],[0],color="#C62828",lw=2,label="bonding CP edge (non-covalent)"),
    Line2D([0],[0],color=CP_LONE_C,lw=1,ls="--",label="lone-pair CP edge"),
    Line2D([0],[0],color=CP_CORE_C,lw=0.8,ls=":",label="core CP edge"),
]
fig.legend(handles=leg, loc="lower center", ncol=4, fontsize=8,
           bbox_to_anchor=(0.5, -0.06))
fig.suptitle(
    f"ELF CP HeteroData — {MOL_ID}  |  {smiles}\n"
    f"atom.x: {list(atom_x.shape)}   cp.x: {list(cp_x.shape)}   "
    f"edge_index: {list(edge_index.shape)}",
    fontsize=10)
plt.tight_layout()
plt.show()


## Final comparison: all three graphs side by side

Now we can directly compare the three graph construction strategies on the same molecule with the same RDKit 2D layout.

In [ ]:
# Final 3-panel: structure | SOTA graph | ELF attractor graph
pos, _ = get_atom_pixel_coords(smiles, n_atoms)
mg  = set(mg_edges); bnd = set(bonds_col)
in_both = sorted(mg & bnd); only_bond = sorted(bnd-mg); only_mg = sorted(mg-bnd)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

mol_image_only(axes[0], smiles, n_atoms,
    title=f"Molecular Structure\n{n_atoms} atoms, {len(mg)} real bonds")

# SOTA
sota_e  = sorted(bonds_col)
sota_c  = ["#2E7D32" if e in mg else "#C62828" for e in sota_e]
sota_ex = [(midpx(pos,i,j), "#2E7D32" if (i,j) in mg else "#C62828", "s", 11)
           for i,j in sota_e]
draw_graph_on_image(axes[1], smiles, n_atoms, syms, sota_e,
    edge_colors=sota_c, edge_widths=[2.5]*len(sota_e), extra_nodes=sota_ex,
    title=f"SOTA graph  (QTAIM BCPs as bond nodes)\n"
          f"{n_atoms} atom + {len(sota_e)} bond nodes  ·  "
          f"{len(in_both)} real, {len(only_bond)} phantom, {len(only_mg)} missing")

# ELF
bond_attr = [m for m in cp_metadata if m["type"]=="valence" and len(m["atoms"])==2]
elf_e  = [(m["atoms"][0], m["atoms"][1]) for m in bond_attr]
elf_c  = ["#2E7D32" if (min(m["atoms"][0],m["atoms"][1]),
                        max(m["atoms"][0],m["atoms"][1])) in mg
           else "#C62828" for m in bond_attr]
elf_ex = [(midpx(pos, m["atoms"][0], m["atoms"][1]), CP_BOND_C, "D", 10)
          for m in bond_attr]
draw_graph_on_image(axes[2], smiles, n_atoms, syms, elf_e,
    edge_colors=elf_c, edge_widths=[2.5]*len(elf_e), extra_nodes=elf_ex,
    title=f"ELF graph  (bonding attractors as nodes)\n"
          f"{n_atoms} atom + {len(elf_e)} attractor nodes  ·  all real bonds covered")

leg = [
    mpatches.Patch(color=ATOM_COLOR["C"],label="C"),
    mpatches.Patch(color=ATOM_COLOR["N"],label="N"),
    mpatches.Patch(color=ATOM_COLOR["O"],label="O"),
    mpatches.Patch(color=ATOM_COLOR["H"],label="H"),
    Line2D([0],[0],color="#2E7D32",lw=2.5,label="real covalent bond"),
    Line2D([0],[0],color="#C62828",lw=2.5,label="phantom / non-bonded"),
    Line2D([0],[0],color="#E65100",lw=2,ls="--",label=f"real bond missing from SOTA ({len(only_mg)})"),
    plt.scatter([],[],marker="s",c="#2E7D32",s=90,label="SOTA bond node (real)"),
    plt.scatter([],[],marker="s",c="#C62828",s=90,label="SOTA bond node (phantom)"),
    plt.scatter([],[],marker="D",c=CP_BOND_C,s=90,label="ELF bonding attractor node"),
]
fig.legend(handles=leg, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5,-0.04))
fig.suptitle(
    f"{MOL_ID}  |  {smiles}\n"
    f"Real bonds: {len(mg)}  ·  "
    f"SOTA bond nodes: {len(sota_e)} ({len(only_bond)} phantom, {len(only_mg)} missing)  ·  "
    f"ELF attractor nodes: {len(elf_e)}",
    fontsize=10)
plt.tight_layout(); plt.show()


## Graph comparison with edge indices

Four panels showing **exactly** why ELF features cannot fully map onto the SOTA graph:

1. Standard molecular graph — all real bonds, labeled edges
2. SOTA graph — QTAIM BCP topology, labeled edges (i→bk)
3. ELF graph — bonding attractor topology, labeled edges (i→ak)
4. Mismatch — which pairs exist in ELF but not SOTA (and vice versa)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Graph comparison with labeled edges
# Shows exactly which atom pairs are connected in each graph
# and WHY some ELF attractor features cannot map onto the SOTA graph
# ══════════════════════════════════════════════════════════════════════════════
import json as _json
from rdkit.Chem import AllChem, Draw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

ATOM_COLOR = {"C":"#2C2C2A","N":"#1E5FA5","O":"#C94F2A","H":"#9E9C96","F":"#1D9E75"}

# ── RDKit 2D positions ────────────────────────────────────────────────────────
def _pos2d(smiles, n):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.Compute2DCoords(mol)
    conf = mol.GetConformer()
    return {i: np.array([conf.GetAtomPosition(i).x,
                          conf.GetAtomPosition(i).y])
            for i in range(n)}

# ── Rebuild ELF bond attractors from JSON ────────────────────────────────────
jf = f"{JSON_DIR}/integrated_aimel_{gdb_num:06d}.json"
with open(jf) as f: _jdata = _json.load(f)
_atom_map = {str(info.get("Atom list","")).strip(): idx
             for idx, (_, info) in enumerate(_jdata["Atoms"].items())}
_bond_attr = []
for _, cp in _jdata["Critical Points"].items():
    raw   = str(cp.get("Atom list","")).strip()
    items = [a.strip() for a in raw.strip("()").split(",") if a.strip()]
    idxs  = [_atom_map[a] for a in items if a in _atom_map]
    if cp.get("Type","") == "valence" and len(idxs) == 2:
        _bond_attr.append(idxs)   # [atom_i, atom_j]

pos_a  = _pos2d(smiles, n_atoms)
mg     = set(mg_edges)
sota_b = sorted(bonds_col)   # QTAIM BCP list

# ── Core drawing function ─────────────────────────────────────────────────────
def draw_graph(ax, atom_pos, syms, n_atoms,
               edges,         # list of (src, dst) where src/dst can be int or str
               node_pos,      # full pos dict including extra nodes
               node_colors,   # dict {node_id: color}
               node_sizes,    # dict {node_id: size}
               node_markers,  # dict {node_id: marker}
               node_labels,   # dict {node_id: label string}
               edge_colors,   # list of colors per edge
               edge_labels,   # list of label strings per edge (e.g. "0->b3")
               title):

    # axis limits
    xs = [p[0] for p in node_pos.values()]
    ys = [p[1] for p in node_pos.values()]
    px = (max(xs)-min(xs))*0.22 + 0.6
    py = (max(ys)-min(ys))*0.22 + 0.6
    ax.set_xlim(min(xs)-px, max(xs)+px)
    ax.set_ylim(min(ys)-py, max(ys)+py)
    ax.set_aspect("equal"); ax.set_facecolor("white"); ax.axis("off")
    ax.set_title(title, fontsize=9, pad=8)

    # Draw edges with labels at midpoint
    for (s, d), ec, elbl in zip(edges, edge_colors, edge_labels):
        x0, y0 = node_pos[s]
        x1, y1 = node_pos[d]
        ax.plot([x0, x1], [y0, y1], color=ec, lw=1.5, zorder=1,
                solid_capstyle="round")
        # Edge label at midpoint, rotated along edge
        mx, my = (x0+x1)/2, (y0+y1)/2
        angle = np.degrees(np.arctan2(y1-y0, x1-x0))
        if angle < -90 or angle > 90: angle += 180
        ax.text(mx, my, elbl, fontsize=4.5, color=ec,
                ha="center", va="bottom", rotation=angle,
                rotation_mode="anchor", zorder=4,
                bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none", alpha=0.7))

    # Draw nodes
    for nid, (nx_, ny) in node_pos.items():
        c  = node_colors.get(nid,  "#888")
        sz = node_sizes.get(nid,   300)
        mk = node_markers.get(nid, "o")
        ax.scatter(nx_, ny, c=c, s=sz, marker=mk, zorder=5,
                   edgecolors="white", linewidths=0.7)
        ax.text(nx_, ny, node_labels.get(nid, str(nid)),
                ha="center", va="center", fontsize=5,
                color="white", fontweight="bold", zorder=6)


# ═════════════════════════════════════════════════════════════════
# PANEL 1: Standard molecular graph  (what every GNN sees)
# Edges: atom_i ↔ atom_j  labeled "i->j"
# ═════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(1, 2, figsize=(16, 7))

img = Draw.MolToImage(Chem.AddHs(Chem.MolFromSmiles(smiles)), size=(500,500))
axes1[0].imshow(img); axes1[0].axis("off")
axes1[0].set_title(f"Molecular Structure\n{smiles}", fontsize=11)

mg_sorted = sorted(mg_edges)
draw_graph(
    axes1[1],
    atom_pos  = pos_a,
    syms      = syms,
    n_atoms   = n_atoms,
    edges     = [(i, j) for i,j in mg_sorted] + [(j, i) for i,j in mg_sorted],
    node_pos  = pos_a,
    node_colors  = {i: ATOM_COLOR.get(syms[i],"#888") for i in range(n_atoms)},
    node_sizes   = {i: 420 if syms[i]!="H" else 160 for i in range(n_atoms)},
    node_markers = {i: "o" for i in range(n_atoms)},
    node_labels  = {i: f"{i}\n{syms[i]}" for i in range(n_atoms)},
    edge_colors  = ["#555555"]*len(mg_sorted)*2,
    edge_labels  = [f"{i}->{j}" for i,j in mg_sorted] +
                   [f"{j}->{i}" for i,j in mg_sorted],
    title = f"Standard molecular graph\n"
            f"{n_atoms} atoms · {len(mg_sorted)} bonds (bidirectional)\n"
            f"edge_index shape: [2, {len(mg_sorted)*2}]"
)
plt.tight_layout(); plt.show()


# ═════════════════════════════════════════════════════════════════
# PANEL 2: SOTA graph  (atom→bond_node→atom)
# Edges: atom_i ↔ bond_node_k  labeled "i->bk"
# ═════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 7))
axes2[0].imshow(img); axes2[0].axis("off")
axes2[0].set_title(f"Molecular Structure\n{smiles}", fontsize=11)

# Node positions
sota_pos = dict(pos_a)
for k, (i,j) in enumerate(sota_b):
    sota_pos[f"b{k}"] = (pos_a[i] + pos_a[j]) / 2

# Edges + labels
sota_edges  = []
sota_ec     = []
sota_elbl   = []
for k, (i,j) in enumerate(sota_b):
    bn  = f"b{k}"
    col = "#2E7D32" if (i,j) in mg else "#C62828"
    sota_edges += [(i, bn), (bn, j)]
    sota_ec    += [col, col]
    sota_elbl  += [f"{i}->b{k}", f"b{k}->{j}"]

# Node dicts
sota_nc = {i: ATOM_COLOR.get(syms[i],"#888") for i in range(n_atoms)}
sota_nc.update({f"b{k}": "#2E7D32" if (i,j) in mg else "#C62828"
                for k,(i,j) in enumerate(sota_b)})
sota_ns = {i: 420 if syms[i]!="H" else 160 for i in range(n_atoms)}
sota_ns.update({f"b{k}": 260 for k in range(len(sota_b))})
sota_nm = {i: "o" for i in range(n_atoms)}
sota_nm.update({f"b{k}": "s" for k in range(len(sota_b))})
sota_nl = {i: f"{i}\n{syms[i]}" for i in range(n_atoms)}
sota_nl.update({f"b{k}": f"b{k}" for k in range(len(sota_b))})

in_both = [e for e in sota_b if e in mg]
phantom = [e for e in sota_b if e not in mg]
missing = [e for e in mg_edges if e not in set(sota_b)]

draw_graph(axes2[1], pos_a, syms, n_atoms,
           sota_edges, sota_pos, sota_nc, sota_ns, sota_nm, sota_nl,
           sota_ec, sota_elbl,
           title=f"SOTA graph (QTAIM BCP topology)\n"
                 f"{n_atoms} atom nodes (●) + {len(sota_b)} bond nodes (■)\n"
                 f"edge_index shape: [2, {len(sota_edges)}]\n"
                 f"{len(in_both)} real · {len(phantom)} phantom · {len(missing)} MISSING")
plt.tight_layout(); plt.show()


# ═════════════════════════════════════════════════════════════════
# PANEL 3: ELF graph  (atom→attractor→atom)
# Edges: atom_i ↔ attractor_k  labeled "i->ak"
# ═════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(16, 7))
axes3[0].imshow(img); axes3[0].axis("off")
axes3[0].set_title(f"Molecular Structure\n{smiles}", fontsize=11)

elf_pos = dict(pos_a)
for k, (i,j) in enumerate(_bond_attr):
    elf_pos[f"a{k}"] = (pos_a[i] + pos_a[j]) / 2

elf_edges = []
elf_ec    = []
elf_elbl  = []
for k, (i,j) in enumerate(_bond_attr):
    ak  = f"a{k}"
    col = "#2E7D32" if (min(i,j),max(i,j)) in mg else "#C62828"
    elf_edges += [(i, ak), (ak, j)]
    elf_ec    += [col, col]
    elf_elbl  += [f"{i}->a{k}", f"a{k}->{j}"]

elf_nc = {i: ATOM_COLOR.get(syms[i],"#888") for i in range(n_atoms)}
elf_nc.update({f"a{k}": "#E8870A" for k in range(len(_bond_attr))})
elf_ns = {i: 420 if syms[i]!="H" else 160 for i in range(n_atoms)}
elf_ns.update({f"a{k}": 260 for k in range(len(_bond_attr))})
elf_nm = {i: "o" for i in range(n_atoms)}
elf_nm.update({f"a{k}": "D" for k in range(len(_bond_attr))})
elf_nl = {i: f"{i}\n{syms[i]}" for i in range(n_atoms)}
elf_nl.update({f"a{k}": f"a{k}" for k in range(len(_bond_attr))})

n_real = sum(1 for i,j in _bond_attr if (min(i,j),max(i,j)) in mg)

draw_graph(axes3[1], pos_a, syms, n_atoms,
           elf_edges, elf_pos, elf_nc, elf_ns, elf_nm, elf_nl,
           elf_ec, elf_elbl,
           title=f"ELF graph (bonding attractors)\n"
                 f"{n_atoms} atom nodes (●) + {len(_bond_attr)} attractor nodes (◆)\n"
                 f"edge_index shape: [2, {len(elf_edges)}]\n"
                 f"{n_real} real bonds · {len(_bond_attr)-n_real} non-covalent")
plt.tight_layout(); plt.show()


# ═════════════════════════════════════════════════════════════════
# PANEL 4: WHY ELF features cannot map onto SOTA
# Side by side: SOTA bond nodes vs ELF attractor nodes
# Highlighted: pairs that exist in ELF but NOT in SOTA
# ═════════════════════════════════════════════════════════════════
fig4, axes4 = plt.subplots(1, 2, figsize=(18, 8))

sota_pairs = set(sota_b)
elf_pairs  = {(min(i,j),max(i,j)) for i,j in _bond_attr}
elf_only   = sorted(elf_pairs - sota_pairs)   # in ELF but not SOTA
sota_only  = sorted(sota_pairs - elf_pairs)   # in SOTA but not ELF
shared     = sorted(elf_pairs & sota_pairs)

# Left: SOTA — highlight missing ELF pairs in orange outline
ax = axes4[0]
sota_edges2  = []; sota_ec2 = []; sota_elbl2 = []
for k, (i,j) in enumerate(sota_b):
    bn = f"b{k}"
    col = "#2E7D32" if (i,j) in mg else "#C62828"
    sota_edges2 += [(i, bn), (bn, j)]
    sota_ec2    += [col, col]
    sota_elbl2  += [f"{i}->b{k}", f"b{k}->{j}"]
draw_graph(ax, pos_a, syms, n_atoms,
           sota_edges2, sota_pos, sota_nc, sota_ns, sota_nm, sota_nl,
           sota_ec2, sota_elbl2,
           title=f"SOTA graph\n"
                 f"Orange arrows = ELF pairs with NO bond node here\n"
                 f"({len(elf_only)} real bonds that ELF covers but SOTA misses)")
# Add annotation arrows for ELF-only pairs
for i,j in elf_only:
    x0, y0 = pos_a[i]; x1, y1 = pos_a[j]
    ax.annotate("", xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle="->", color="#FF6600",
                                lw=2, connectionstyle="arc3,rad=0.3"))
    mx, my = (x0+x1)/2+0.15, (y0+y1)/2+0.15
    ax.text(mx, my, f"NO node\n({i},{j})\n{syms[i]}-{syms[j]}",
            fontsize=5, color="#FF6600", ha="center",
            bbox=dict(boxstyle="round", fc="white", ec="#FF6600", alpha=0.85))

# Right: ELF — highlight SOTA-only (phantom) pairs
ax = axes4[1]
draw_graph(ax, pos_a, syms, n_atoms,
           elf_edges, elf_pos, elf_nc, elf_ns, elf_nm, elf_nl,
           elf_ec, elf_elbl,
           title=f"ELF graph\n"
                 f"Red arrows = SOTA phantom BCPs with NO attractor in ELF\n"
                 f"({len(sota_only)} QTAIM BCPs that SOTA has but ELF doesn't)")
for i,j in sota_only:
    x0, y0 = pos_a[i]; x1, y1 = pos_a[j]
    ax.annotate("", xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle="->", color="#C62828",
                                lw=2, connectionstyle="arc3,rad=0.3"))
    mx, my = (x0+x1)/2-0.15, (y0+y1)/2+0.15
    ax.text(mx, my, f"NO attractor\n({i},{j})\n{syms[i]}-{syms[j]}",
            fontsize=5, color="#C62828", ha="center",
            bbox=dict(boxstyle="round", fc="white", ec="#C62828", alpha=0.85))

leg4 = [
    mpatches.Patch(color=ATOM_COLOR["C"],label="C"),
    mpatches.Patch(color=ATOM_COLOR["N"],label="N"),
    mpatches.Patch(color=ATOM_COLOR["O"],label="O"),
    mpatches.Patch(color=ATOM_COLOR["H"],label="H"),
    mpatches.Patch(color="#2E7D32",label="shared (real bond in both)"),
    mpatches.Patch(color="#C62828",label="SOTA phantom BCP (not in ELF)"),
    mpatches.Patch(color="#E8870A",label="ELF attractor node"),
    Line2D([0],[0],color="#FF6600",lw=2,label=f"ELF pair with NO bond node in SOTA ({len(elf_only)})"),
    Line2D([0],[0],color="#C62828",lw=2,label=f"SOTA phantom BCP not in ELF ({len(sota_only)})"),
]
fig4.legend(handles=leg4, loc="lower center", ncol=3, fontsize=8,
            bbox_to_anchor=(0.5,-0.04))
fig4.suptitle(
    f"{MOL_ID}  |  {smiles}\n"
    f"Shared pairs: {len(shared)}  ·  "
    f"ELF-only (real bonds SOTA misses): {len(elf_only)}  ·  "
    f"SOTA phantom BCPs (not in ELF): {len(sota_only)}",
    fontsize=10)
plt.tight_layout(); plt.show()

# Print the mismatch table
print("\n" + "="*65)
print("  WHY ELF features cannot fully map onto the SOTA graph")
print("="*65)
print(f"\n  ELF has attractor for {len(elf_pairs)} atom pairs")
print(f"  SOTA has bond node for {len(sota_pairs)} atom pairs")
print(f"  Shared (both have a node)    : {len(shared)}")
print(f"  ELF-only — REAL bonds SOTA MISSES: {len(elf_only)}")
for i,j in elf_only:
    print(f"    ({i:2d},{j:2d})  {syms[i]}-{syms[j]}  ← ELF has attractor here, SOTA has NO bond node")
print(f"  SOTA phantom BCPs not in ELF : {len(sota_only)}")
for i,j in sota_only:
    print(f"    ({i:2d},{j:2d})  {syms[i]}-{syms[j]}  ← SOTA has bond node here, ELF has no attractor")
print("="*65)


## Why does QTAIM miss real bonds?

QTAIM defines a bond by a **Bond Critical Point (BCP)** — a saddle point of ρ(r). It fails for:
- **Aromatic ring bonds** — ring and bond CPs can nearly merge
- **Polar bonds (N–O, C–O)** — very asymmetric density makes BCP hard to locate
- **C–H / N–H bonds** — density extremely asymmetric near H
- **Numerical issues** — finite DFT grid misses flat BCPs

QTAIM also **adds phantom BCPs** for non-bonded through-space interactions (H···H, N···C) where atoms are close in space.

**For your thesis:** The SOTA graph topology is incomplete and noisy as a molecular representation. ELF attractors cover all real bonds because they represent electron pair domains — a more robust quantum chemical basis for graph construction.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Why does QTAIM miss real bonds? — analysis for this specific molecule
# ══════════════════════════════════════════════════════════════════════════════
import json as _json
from rdkit.Chem import AllChem, Draw
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

ATOM_COLOR = {"C":"#2C2C2A","N":"#1E5FA5","O":"#C94F2A","H":"#9E9C96","F":"#1D9E75"}

def _pos2d(smiles, n):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.Compute2DCoords(mol)
    conf = mol.GetConformer()
    return {i: np.array([conf.GetAtomPosition(i).x,
                          conf.GetAtomPosition(i).y])
            for i in range(n)}

# Rebuild bond attractor list
jf = f"{JSON_DIR}/integrated_aimel_{gdb_num:06d}.json"
with open(jf) as f: _jdata = _json.load(f)
_atom_map = {str(info.get("Atom list","")).strip(): idx
             for idx, (_, info) in enumerate(_jdata["Atoms"].items())}
elf_pairs = set()
for _, cp in _jdata["Critical Points"].items():
    raw   = str(cp.get("Atom list","")).strip()
    items = [a.strip() for a in raw.strip("()").split(",") if a.strip()]
    idxs  = [_atom_map[a] for a in items if a in _atom_map]
    if cp.get("Type","") == "valence" and len(idxs) == 2:
        i, j = idxs
        elf_pairs.add((min(i,j), max(i,j)))

pos_a      = _pos2d(smiles, n_atoms)
mg         = set(mg_edges)
sota_pairs = set(bonds_col)

missing_from_sota = sorted(mg - sota_pairs)    # real bonds, no QTAIM BCP
phantom_in_sota   = sorted(sota_pairs - mg)    # QTAIM BCPs, not real bonds
covered_by_both   = sorted(mg & sota_pairs)

# ── Print detailed analysis ───────────────────────────────────────────────────
print(f"Molecule: {MOL_ID}  |  SMILES: {smiles}")
print(f"Atoms: {n_atoms}  |  Real bonds: {len(mg)}")
print()
print(f"QTAIM bond nodes: {len(sota_pairs)}")
print(f"  Covers {len(covered_by_both)}/{len(mg)} real bonds  ({100*len(covered_by_both)/len(mg):.0f}%)")
print(f"  Misses {len(missing_from_sota)} real bonds")
print(f"  Adds   {len(phantom_in_sota)} phantom (non-covalent) BCPs")
print()
print("Real bonds MISSING from QTAIM graph (no bond node for these pairs):")
print(f"  {'pair':8}  {'atoms':8}  {'likely reason'}")
print(f"  {'─'*60}")
for i,j in missing_from_sota:
    si, sj = syms[i], syms[j]
    pair_type = f"{si}-{sj}"
    # Classify the likely reason
    # Check if both atoms are in a ring
    from rdkit import Chem
    mol_check = Chem.MolFromSmiles(smiles)
    ri = mol_check.GetRingInfo()
    ring_atoms = set(a for r in ri.AtomRings() for a in r)
    # Note: heavy atom indices differ from pymatgen (no H), but ring pattern still informative
    if pair_type in ["N-O","O-N","C-O","O-C","C-N","N-C"] and i < 8 and j < 8:
        reason = "polar bond in aromatic ring — BCP hard to locate"
    elif pair_type in ["C-H","N-H","H-C","H-N"]:
        reason = "C/N-H bond — density very asymmetric near H"
    elif pair_type in ["C-C","C=C"]:
        reason = "C-C in ring — ring/bond CP near degenerate"
    else:
        reason = "BCP not detected numerically"
    print(f"  ({i:2d},{j:2d})   {pair_type:<8}  {reason}")

print()
print("Phantom QTAIM BCPs (non-bonded through-space interactions):")
print(f"  {'pair':8}  {'atoms':8}  {'likely reason'}")
print(f"  {'─'*60}")
for i,j in phantom_in_sota:
    si, sj = syms[i], syms[j]
    pair_type = f"{si}-{sj}"
    if "H" in pair_type:
        reason = "H···H or X···H van der Waals / H-bond interaction"
    else:
        reason = "through-space non-covalent BCP (close distance)"
    print(f"  ({i:2d},{j:2d})   {pair_type:<8}  {reason}")

# ── Figure: the three bond categories visualised ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

img = Draw.MolToImage(
    Chem.AddHs(Chem.MolFromSmiles(smiles)), size=(500,500))

def _draw_bonds(ax, bond_list, ec, ls, lw, label_suffix=""):
    for i, j in bond_list:
        x0, y0 = pos_a[i]; x1, y1 = pos_a[j]
        ax.plot([x0,x1],[y0,y1], color=ec, lw=lw, ls=ls, zorder=1,
                solid_capstyle="round")
        mx, my = (x0+x1)/2, (y0+y1)/2
        angle  = np.degrees(np.arctan2(y1-y0, x1-x0))
        if angle < -90 or angle > 90: angle += 180
        ax.text(mx, my, f"({i},{j}){label_suffix}",
                fontsize=4.5, color=ec, ha="center", va="bottom",
                rotation=angle, rotation_mode="anchor", zorder=4,
                bbox=dict(boxstyle="round,pad=0.05",fc="white",ec="none",alpha=0.7))

def _draw_atoms(ax):
    xs = [pos_a[i][0] for i in range(n_atoms)]
    ys = [pos_a[i][1] for i in range(n_atoms)]
    xp = (max(xs)-min(xs))*0.22+0.6
    yp = (max(ys)-min(ys))*0.22+0.6
    ax.set_xlim(min(xs)-xp, max(xs)+xp)
    ax.set_ylim(min(ys)-yp, max(ys)+yp)
    ax.set_aspect("equal"); ax.set_facecolor("white"); ax.axis("off")
    for i in range(n_atoms):
        c  = ATOM_COLOR.get(syms[i],"#888")
        sz = 420 if syms[i]!="H" else 160
        ax.scatter(pos_a[i][0], pos_a[i][1], c=c, s=sz, zorder=3,
                   edgecolors="white", linewidths=0.7)
        ax.text(pos_a[i][0], pos_a[i][1], f"{i}\n{syms[i]}",
                ha="center", va="center", fontsize=5.5,
                color="white", fontweight="bold", zorder=4)

# Panel 1: All real bonds — ground truth
_draw_bonds(axes[0], sorted(mg), "#555555", "-", 2.0)
_draw_atoms(axes[0])
axes[0].set_title(f"All real bonds ({len(mg)})\n"
                  f"molecule_graph.graph.edges()\nThis is ground truth",
                  fontsize=10)

# Panel 2: SOTA — coloured by category
_draw_bonds(axes[1], covered_by_both, "#2E7D32", "-",  2.2, " ✓")
_draw_bonds(axes[1], missing_from_sota, "#E65100", "--", 2.0, " ✗no BCP")
_draw_bonds(axes[1], phantom_in_sota,   "#C62828", ":",  2.0, " phantom")
_draw_atoms(axes[1])
axes[1].set_title(f"QTAIM topology\n"
                  f"green=has bond node ({len(covered_by_both)})\n"
                  f"orange--=MISSING ({len(missing_from_sota)})  ·  red:=phantom ({len(phantom_in_sota)})",
                  fontsize=10)

# Panel 3: ELF — all real bonds covered
_draw_bonds(axes[2], sorted(elf_pairs & mg),  "#2E7D32", "-", 2.2, " ✓")
_draw_bonds(axes[2], sorted(elf_pairs - mg),  "#C62828", ":", 2.0, " non-cov")
_draw_bonds(axes[2], sorted(mg - elf_pairs),  "#E65100", "--",2.0, " ✗no attr")
_draw_atoms(axes[2])
axes[2].set_title(f"ELF attractor topology\n"
                  f"green=has attractor ({len(elf_pairs & mg)})\n"
                  f"orange--=missing ({len(mg-elf_pairs)})  ·  red:=non-cov ({len(elf_pairs-mg)})",
                  fontsize=10)

leg = [
    Line2D([0],[0],color="#2E7D32",lw=2,     label="bond in graph ✓"),
    Line2D([0],[0],color="#E65100",lw=2,ls="--",label="real bond MISSING from graph ✗"),
    Line2D([0],[0],color="#C62828",lw=2,ls=":", label="phantom/non-covalent BCP"),
]
fig.legend(handles=leg, loc="lower center", ncol=3, fontsize=10,
           bbox_to_anchor=(0.5,-0.02))
fig.suptitle(
    f"{MOL_ID}  |  {smiles}\n"
    f"QTAIM covers {len(covered_by_both)}/{len(mg)} real bonds — "
    f"misses {len(missing_from_sota)}, adds {len(phantom_in_sota)} phantoms\n"
    f"ELF covers {len(elf_pairs & mg)}/{len(mg)} real bonds — "
    f"misses {len(mg-elf_pairs)}, adds {len(elf_pairs-mg)} non-covalent",
    fontsize=10)
plt.tight_layout(); plt.show()
